# S4 Demo — Deploy the Session 2 Model (Topic 4.2)
`models/churn_xgb_v1.joblib` — trained in Session 2 — becomes a real-time endpoint, gets invoked, and is
**deleted live** (the most important cell). Pre-deploy ~15 min before class so it's `InService`.
Local twin (no AWS): `uvicorn session4.local_api:app` — same pattern, zero cloud.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'src').exists():
    p = p.parent
sys.path.insert(0, str(p / 'src')); REPO = p
import warnings; warnings.filterwarnings('ignore')

In [ ]:
from course_utils import models
model = models.load_model('churn_xgb_v1')
print(type(model).__name__, '| features:', len(getattr(model, 'feature_names_in_', [])))

## Deploy (run pre-class — takes ~5 min to reach InService)
```python
from course_utils import aws
predictor = aws.deploy_sklearn_model('models/churn_xgb_v1.joblib', endpoint_name='mlcourse-churn')
```
While it spins up, show the console: endpoint status, instance type, **the hourly meter**.

## Invoke — one customer in, probability out (~50 ms)

In [ ]:
from course_utils import data, aws
df = data.load_processed('telco_clean')
X, y = data.prep_features(df)
sample = X.iloc[[0]].astype(float).to_dict(orient='records')

TRY_AWS = False          # flip in class when the endpoint is InService
if TRY_AWS:
    print(aws.invoke_endpoint('mlcourse-churn', {'instances': sample}))
else:
    print({'churn_probability': [round(float(model.predict_proba(X.iloc[[0]])[0, 1]), 4)],
           'note': 'local twin - identical contract'})

## Latency reality check (batch of 50)

In [ ]:
import time, numpy as np
lat = []
for i in range(50):
    t0 = time.perf_counter()
    _ = model.predict_proba(X.iloc[[i % len(X)]])
    lat.append((time.perf_counter() - t0) * 1000)
print(f'local p50={np.percentile(lat, 50):.1f} ms  p95={np.percentile(lat, 95):.1f} ms  '
      '(endpoint adds network + serialization overhead)')

## THE most important cell — delete it, live, narrated
```python
aws.delete_endpoint('mlcourse-churn')
```
Endpoints bill **while idle**. Cost hygiene is a graded habit (see syllabus).